In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
from dotenv import load_dotenv

In [4]:
load_dotenv()

True

In [5]:
llm =ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [6]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [7]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}


In [8]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [9]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza get a job?\n\nBecause it kneaded the dough!',
 'explanation': 'This is a classic pun! The humor comes from two sets of wordplay:\n\n1.  **"Kneaded" vs. "Needed":** The word "kneaded" (as in, working dough to make bread or pizza) sounds exactly like "needed" (as in, requiring something).\n\n2.  **"Dough" (pizza) vs. "Dough" (money):** "Dough" is the main ingredient of pizza crust. However, "dough" is also a common slang term for "money."\n\nSo, the joke works because:\n\n*   **Literally, for a pizza:** A pizza is made from dough, and that dough is *kneaded* during preparation.\n*   **Figuratively, for a person getting a job:** Someone gets a job because they *need* *money* (or \'dough\').\n\nThe joke personifies the pizza, giving it a human motivation (needing money) but using words that cleverly relate back to its literal form (pizza dough). The unexpected twist of the financial meaning hidden within the pizza-making term is what makes it 

In [10]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job?\n\nBecause it kneaded the dough!', 'explanation': 'This is a classic pun! The humor comes from two sets of wordplay:\n\n1.  **"Kneaded" vs. "Needed":** The word "kneaded" (as in, working dough to make bread or pizza) sounds exactly like "needed" (as in, requiring something).\n\n2.  **"Dough" (pizza) vs. "Dough" (money):** "Dough" is the main ingredient of pizza crust. However, "dough" is also a common slang term for "money."\n\nSo, the joke works because:\n\n*   **Literally, for a pizza:** A pizza is made from dough, and that dough is *kneaded* during preparation.\n*   **Figuratively, for a person getting a job:** Someone gets a job because they *need* *money* (or \'dough\').\n\nThe joke personifies the pizza, giving it a human motivation (needing money) but using words that cleverly relate back to its literal form (pizza dough). The unexpected twist of the financial meaning hidden within the pizza-making ter

In [11]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a job?\n\nBecause it kneaded the dough!', 'explanation': 'This is a classic pun! The humor comes from two sets of wordplay:\n\n1.  **"Kneaded" vs. "Needed":** The word "kneaded" (as in, working dough to make bread or pizza) sounds exactly like "needed" (as in, requiring something).\n\n2.  **"Dough" (pizza) vs. "Dough" (money):** "Dough" is the main ingredient of pizza crust. However, "dough" is also a common slang term for "money."\n\nSo, the joke works because:\n\n*   **Literally, for a pizza:** A pizza is made from dough, and that dough is *kneaded* during preparation.\n*   **Figuratively, for a person getting a job:** Someone gets a job because they *need* *money* (or \'dough\').\n\nThe joke personifies the pizza, giving it a human motivation (needing money) but using words that cleverly relate back to its literal form (pizza dough). The unexpected twist of the financial meaning hidden within the pizza-making te

In [24]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f16aea3-8e1c-6c48-8000-083a398678cf"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f16aea3-8e1c-6c48-8000-083a398678cf'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-06-18T07:49:23.440954+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f16aea3-8e1a-6537-bfff-a80de2ac693f'}}, tasks=(PregelTask(id='fd168877-127c-21e5-8589-f6349e6d6105', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza get a job?\n\nBecause it kneaded the dough!'}),), interrupts=())

In [25]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f16aea3-8e1c-6c48-8000-083a398678cf"}})

{'topic': 'pizza',
 'joke': 'Why did the pizza get a ticket?\n\nBecause it was *speeding* to get to my house!',
 'explanation': 'This joke is a classic example of a **pun**! It plays on the double meaning of the word "speeding."\n\nHere\'s the breakdown:\n\n1.  **The Literal Meaning:** When you hear "get a ticket" and "speeding," your mind immediately goes to a car or vehicle driving too fast and breaking traffic laws. People get tickets for literal "speeding."\n\n2.  **The Pun/Figurative Meaning:** In the context of pizza, "speeding" refers to the *speed of delivery*. Everyone wants their pizza to arrive quickly, or "speedily," so it\'s still hot and fresh. The pizza\'s "mission" is to get to your house as fast as possible.\n\nThe humor comes from the **absurdity and personification**. It creates a silly image of the pizza itself being the one in a hurry (and therefore deserving of a ticket for being so fast), rather than the delivery driver.\n\nSo, the pizza "got a ticket" not for a 

In [26]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a ticket?\n\nBecause it was *speeding* to get to my house!', 'explanation': 'This joke is a classic example of a **pun**! It plays on the double meaning of the word "speeding."\n\nHere\'s the breakdown:\n\n1.  **The Literal Meaning:** When you hear "get a ticket" and "speeding," your mind immediately goes to a car or vehicle driving too fast and breaking traffic laws. People get tickets for literal "speeding."\n\n2.  **The Pun/Figurative Meaning:** In the context of pizza, "speeding" refers to the *speed of delivery*. Everyone wants their pizza to arrive quickly, or "speedily," so it\'s still hot and fresh. The pizza\'s "mission" is to get to your house as fast as possible.\n\nThe humor comes from the **absurdity and personification**. It creates a silly image of the pizza itself being the one in a hurry (and therefore deserving of a ticket for being so fast), rather than the delivery driver.\n\nSo, the pizza "got 

In [27]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f16aeb5-9f84-6bf9-8000-4ced20f4eaa8'}}

In [28]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f16aeb5-9f84-6bf9-8000-4ced20f4eaa8'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-06-18T07:57:28.449944+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='f9f43c07-64a1-0a55-ba5c-618c1a209933', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a ticket?\n\nBecause it was *speeding* to get to my house!', 'explanation': 'This joke is a classic example of a **pun**! It plays on the double meaning of the word "speeding."\n\nHere\'s the breakdown:\n\n1.  **The Literal Meaning:** When you hear "get a ticket" and "speeding," your mind immediately goes t

In [29]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f16aea5-b075-620a-8000-77449873742b"}})

{'topic': 'samosa',
 'joke': 'Why did the samosa go to therapy?\n\nBecause it had too many **filling** issues!',
 'explanation': 'This joke is a classic pun that plays on the similar sounds of two different words: "filling" and "feeling."\n\nHere\'s the breakdown:\n\n1.  **For a Samosa (Literal Meaning):** A samosa is a pastry filled with savory ingredients like potatoes, peas, and spices. So, a samosa literally has **filling**. "Filling issues" could humorously refer to problems with its contents – maybe not enough filling, bad filling, or the filling spilling out.\n\n2.  **For a Person (Figurative Meaning/Therapy Context):** When people go to therapy, they often do so because they are experiencing "feeling issues." This refers to emotional problems, difficulties processing or expressing emotions, feeling overwhelmed, anxious, depressed, or generally having trouble with their inner emotional state.\n\n3.  **The Pun:** The humor comes from the fact that "filling issues" (what a samosa 

In [30]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa go to therapy?\n\nBecause it had too many **filling** issues!', 'explanation': 'This joke is a classic pun that plays on the similar sounds of two different words: "filling" and "feeling."\n\nHere\'s the breakdown:\n\n1.  **For a Samosa (Literal Meaning):** A samosa is a pastry filled with savory ingredients like potatoes, peas, and spices. So, a samosa literally has **filling**. "Filling issues" could humorously refer to problems with its contents – maybe not enough filling, bad filling, or the filling spilling out.\n\n2.  **For a Person (Figurative Meaning/Therapy Context):** When people go to therapy, they often do so because they are experiencing "feeling issues." This refers to emotional problems, difficulties processing or expressing emotions, feeling overwhelmed, anxious, depressed, or generally having trouble with their inner emotional state.\n\n3.  **The Pun:** The humor comes from the fact that "filling iss

In [19]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [20]:
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [21]:
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [22]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [23]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...


EmptyInputError: Received no input for __start__

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

NameError: name 'graph' is not defined